# H15a: Direction Quality in a Toy Deep-Linear Setting

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/H15a_DIRECTION_QUALITY_METRIC/run_experiment.py`

This notebook is the explanatory / analysis counterpart to the script above. It is intentionally built to **import and use the script's `run_experiment()` results** rather than re-implement the core experiment logic. The goal of this first completion pass is parity plus clarity: keep the toy experiment intact, make the reporting more truthful, and present the evidence in a more academically serious way.

## Scientific scope

We measure how closely several optimizer step directions align with a local Newton direction in a tiny 2-layer deep linear network (32 total scalar parameters). The model is intentionally small so that a full Hessian can be estimated by central finite differences.


## What this notebook does and does not test

### What it does test
- Local cosine alignment between each optimizer's step direction and a **finite-difference pseudoinverse-Newton direction**.
- The comparison at steps `10, 20, ..., 200` along each optimizer's own trajectory.
- Learning-rate selection by **best final loss within a finite sweep grid** on the same toy training problem.

### What it does **not** establish
- It does **not** prove Muon's mechanism in general.
- It does **not** show an exact Newton direction; the Hessian is finite-difference estimated and then pseudoinverted.
- It does **not** remove every LR-selection confound; several best LRs can sit on the sweep boundary.
- It does **not** isolate step-map rotation from trajectory effects, because each optimizer is evaluated on its own trajectory.
- It does **not** say anything decisive about large nonlinear networks.

So the right interpretation is: **toy-setting directional evidence**, not a general mechanistic proof.


## Imports, explicit path handling, and script loading


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = lambda s: s
    def display(obj):
        print(obj)


plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RELATIVE_SCRIPT = Path('experiments/Tier_1_Core_Mechanism_Tests/H15a_DIRECTION_QUALITY_METRIC/run_experiment.py')


def locate_h15a_script(start_dir: Path):
    start_dir = start_dir.resolve()
    for candidate in [start_dir] + list(start_dir.parents):
        script_path = candidate / RELATIVE_SCRIPT
        if script_path.exists():
            return candidate, script_path.parent, script_path
    raise FileNotFoundError(
        f'Could not locate {RELATIVE_SCRIPT} by walking upward from {start_dir}. '
        'Run this notebook from somewhere inside the project tree.'
    )


PROJECT_ROOT, EXPERIMENT_DIR, SCRIPT_PATH = locate_h15a_script(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

spec = importlib.util.spec_from_file_location('h15a_direction_quality_metric', SCRIPT_PATH)
h15a = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(h15a)

print(f'Project root   : {PROJECT_ROOT}')
print(f'Experiment dir : {EXPERIMENT_DIR}')
print(f'Script path    : {SCRIPT_PATH}')
print("Notebook role  : analysis/reporting layer that consumes the script's returned results")


## Reproducibility, configuration, and runtime expectations


In [ ]:
config_preview = {
    'python_version': platform.python_version(),
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'matplotlib_version': plt.matplotlib.__version__,
    'script_path': str(SCRIPT_PATH),
    'seeds': h15a.make_seeds(h15a.NUM_SEEDS),
    'measure_steps': list(h15a.MEASURE_STEPS),
    'lr_selection_label': 'best_grid_lr',
    'lr_selection_horizon_steps': max(h15a.MEASURE_STEPS),
    'fd_eps': h15a.FD_EPS,
    'newton_pinv_rcond': h15a.NEWTON_PINV_RCOND,
    'batch_size': h15a.BATCH_SIZE,
    'network': f'{h15a.NUM_LAYERS}-layer deep linear, {h15a.DIM}x{h15a.DIM}, {h15a.TOTAL_PARAMS} parameters',
}

display(pd.Series(config_preview, name='value').to_frame())

lr_grid_df = pd.DataFrame({
    name: pd.Series(grid) for name, (_, grid) in h15a.OPTIMIZERS.items()
})
print('Learning-rate grids used by the script (reported LRs are best within these grids):')
display(lr_grid_df)


## Execute the script-backed experiment

The cell below calls `run_experiment(verbose=False)` from the script. This preserves notebook/script parity: the notebook does not maintain a separate copy of the experimental core.


In [ ]:
start = time.time()
results = h15a.run_experiment(verbose=False)
notebook_elapsed = time.time() - start

print(f"Script-reported runtime : {results['runtime_sec']:.2f}s")
print(f"Notebook wall time      : {notebook_elapsed:.2f}s")
print(f"Measurement records     : {len(results['records'])}")
print(f"LR selections           : {len(results['best_grid_lr_records'])}")
print(f"Training runs tracked   : {len(results['training_run_records'])}")


## Convert structured results into analysis tables


In [ ]:
summary_df = (
    pd.DataFrame.from_dict(results['summary_by_optimizer'], orient='index')
    .reset_index(drop=True)
    .sort_values('optimizer')
)
records_df = pd.DataFrame(results['records'])
per_step_df = pd.DataFrame(results['per_step_summary'])
seed_summary_df = pd.DataFrame(results['seed_level_summary'])
lr_df = pd.DataFrame(results['best_grid_lr_records'])
training_df = pd.DataFrame(results['training_run_records'])

paired_rows = []
for baseline, payload in results['paired_differences'].items():
    for row in payload['records']:
        paired_rows.append(dict(row))
paired_df = pd.DataFrame(paired_rows)

hypothesis_rows = []
for test_name, row in results['hypothesis_tests'].items():
    hypothesis_rows.append(dict(row))
hypothesis_df = pd.DataFrame(hypothesis_rows).sort_values('test')

print('Per-optimizer pooled summary:')
display(summary_df[['optimizer', 'n_measurements', 'mean_cos_newton', 'std_cos_newton', 'mean_cos_grad', 'std_cos_grad', 'mean_best_grid_lr', 'hit_grid_min_count', 'hit_grid_max_count']])


## Best-in-grid learning rates by optimizer and seed

This table is intentionally labeled **best-in-grid LR** rather than "optimal LR." Boundary hits matter: if the selected LR equals the minimum or maximum of the sweep range, the true best LR may lie outside the scanned grid.


In [ ]:
def boundary_label(row):
    labels = []
    if row['hit_grid_min']:
        labels.append('MIN')
    if row['hit_grid_max']:
        labels.append('MAX')
    return '+'.join(labels) if labels else ''


lr_df = lr_df.copy()
lr_df['boundary'] = lr_df.apply(boundary_label, axis=1)
lr_df['best_grid_lr_display'] = lr_df.apply(
    lambda row: f"{row['best_grid_lr']:.6f}" + (f" [{row['boundary']}]" if row['boundary'] else ''),
    axis=1,
)

lr_table = (
    lr_df[['seed', 'optimizer', 'best_grid_lr_display']]
    .pivot(index='seed', columns='optimizer', values='best_grid_lr_display')
    .sort_index()
)

boundary_summary = (
    lr_df.groupby('optimizer')[['hit_grid_min', 'hit_grid_max']]
    .sum()
    .astype(int)
    .rename(columns={'hit_grid_min': 'count_hit_grid_min', 'hit_grid_max': 'count_hit_grid_max'})
)

display(lr_table)
print('Boundary-hit counts across seeds:')
display(boundary_summary)


## Per-step trajectory of `cos(step, Newton)`

The figure below shows the main quantity of interest. Each curve is the mean over seeds at a fixed measurement step; the shaded band is **±1 standard deviation across seeds** at that step. This remains a toy-setting plot because each optimizer is evaluated along its own trajectory.


In [ ]:
optimizer_order = list(h15a.OPTIMIZERS.keys())
colors = dict(zip(optimizer_order, plt.rcParams['axes.prop_cycle'].by_key()['color'][:len(optimizer_order)]))

fig, ax = plt.subplots(figsize=(10, 5))
for optimizer in optimizer_order:
    sub = per_step_df[per_step_df['optimizer'] == optimizer].sort_values('step')
    x = sub['step'].to_numpy()
    y = sub['mean_cos_newton'].to_numpy()
    s = sub['std_cos_newton'].fillna(0.0).to_numpy()
    ax.plot(x, y, marker='o', linewidth=2, label=optimizer, color=colors[optimizer])
    ax.fill_between(x, y - s, y + s, alpha=0.18, color=colors[optimizer])

ax.set_title('Per-step direction quality: cos(step, Newton)\nMean ± 1 SD across seeds')
ax.set_xlabel('Training step')
ax.set_ylabel('cos(step, Newton)')
ax.set_ylim(-0.1, 1.05)
ax.legend()
plt.show()


In [ ]:
t1 = results['hypothesis_tests']['T1']
t2 = results['hypothesis_tests']['T2']
display(Markdown(
    f"**Interpretation.** In this deterministic toy setup, Muon can be compared against SGD and the other baselines by tracking how Newton-aligned its step direction remains through training. "
    f"The Muon-vs-SGD comparison is currently labeled **{t1['evidence_label']}**, while the Muon-vs-NormSGD comparison is **{t2['evidence_label']}**. "
    "That is the key reason the notebook should not claim that Muon > NormSGD is fully settled from the present metric alone."
))


## Per-step trajectory of `cos(step, -grad)`

This secondary plot shows how closely each optimizer follows plain steepest descent. A lower value means more deviation from `-gradient`. The relevant story here is not "deviation is always good"; it is whether a deviation from `-gradient` coincides with stronger Newton alignment.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for optimizer in optimizer_order:
    sub = per_step_df[per_step_df['optimizer'] == optimizer].sort_values('step')
    x = sub['step'].to_numpy()
    y = sub['mean_cos_grad'].to_numpy()
    s = sub['std_cos_grad'].fillna(0.0).to_numpy()
    ax.plot(x, y, marker='o', linewidth=2, label=optimizer, color=colors[optimizer])
    ax.fill_between(x, y - s, y + s, alpha=0.18, color=colors[optimizer])

ax.set_title('Per-step steepest-descent similarity: cos(step, -grad)\nMean ± 1 SD across seeds')
ax.set_xlabel('Training step')
ax.set_ylabel('cos(step, -grad)')
ax.set_ylim(-0.1, 1.05)
ax.legend()
plt.show()


## Seed-level summary and Muon-minus-baseline differences

Pooled means are useful, but they are not the whole story. The tables below expose the seed-level mean `cos(step, Newton)` values and the paired Muon-minus-baseline differences. This is especially important for the Muon-vs-NormSGD comparison, where the sign pattern may be mixed even if the pooled mean is slightly positive.


In [ ]:
seed_level_pivot = (
    seed_summary_df[['seed', 'optimizer', 'mean_cos_newton']]
    .pivot(index='seed', columns='optimizer', values='mean_cos_newton')
    .sort_index()
)
paired_summary_rows = []
for baseline, payload in results['paired_differences'].items():
    summary = payload['summary']
    paired_summary_rows.append({
        'comparison': f"Muon_k5 - {baseline}",
        'mean_delta': summary['mean'],
        'std_delta': summary['std'],
        'sem_delta': summary['sem'],
        'positive_seeds': summary['positive_count'],
        'negative_seeds': summary['negative_count'],
        'n_seeds': summary['n'],
    })
paired_summary_df = pd.DataFrame(paired_summary_rows)
paired_seed_pivot = (
    paired_df[['seed', 'baseline', 'muon_minus_baseline']]
    .pivot(index='seed', columns='baseline', values='muon_minus_baseline')
    .sort_index()
)

print('Seed-level mean cos(step, Newton) by optimizer:')
display(seed_level_pivot)
print('Seed-level Muon-minus-baseline differences:')
display(paired_seed_pivot)
print('Paired-difference summaries across seeds:')
display(paired_summary_df)


## T1 / T2 / T3 verdicts with caveats

These are still the original pairwise hypotheses, but they are now presented with both pooled means and seed-level paired differences. The important change is interpretive honesty: a positive pooled delta alone should not be described as fully establishing the comparison if the seed-level evidence is weak or mixed.


In [ ]:
verdict_table = hypothesis_df[[
    'test', 'baseline', 'pooled_mean_muon', 'pooled_mean_baseline', 'pooled_delta',
    'pooled_pass', 'seed_level_mean_delta', 'seed_level_std_delta', 'seed_level_sem_delta',
    'seed_level_positive_count', 'seed_level_negative_count', 'seed_level_n', 'evidence_label'
]].copy()

display(verdict_table)

boundary_counts = lr_df.groupby('optimizer')[['hit_grid_min', 'hit_grid_max']].sum().astype(int)
sgd_max_hits = int(boundary_counts.loc['SGD', 'hit_grid_max']) if 'SGD' in boundary_counts.index else 0
muon_max_hits = int(boundary_counts.loc['Muon_k5', 'hit_grid_max']) if 'Muon_k5' in boundary_counts.index else 0
adam_test = results['hypothesis_tests']['T3']

display(Markdown(f'''### Reading the verdicts responsibly

- **T1 (Muon vs SGD):** `{t1['evidence_label']}`. This is the clearest positive result in the current toy setup.
- **T2 (Muon vs NormSGD):** `{t2['evidence_label']}`. This should be treated as the weakest comparison in the present default configuration, not as a settled win.
- **T3 (Muon vs AdamLike):** `{adam_test['evidence_label']}`.
- **LR-grid caveat:** boundary hits are present (for example, SGD max-grid hits = `{sgd_max_hits}`, Muon max-grid hits = `{muon_max_hits}`), so "best-in-grid" must not be described as a true unconstrained optimum.
- **Toy-scope caveat:** this remains a 32-parameter deep-linear finite-difference study, not a large-scale nonlinear training result.
'''))


## Conclusion


In [ ]:
muon = results['summary_by_optimizer']['Muon_k5']
sgd = results['summary_by_optimizer']['SGD']
norm = results['summary_by_optimizer']['NormSGD']
adam = results['summary_by_optimizer']['AdamLike']

display(Markdown(f'''Under the **default deterministic H15a toy setup**, the evidence is best summarized as follows:

1. **Strongest support:** Muon is more Newton-aligned than plain SGD in this toy setting (`mean cos(step, Newton)` {muon['mean_cos_newton']:.3f} vs {sgd['mean_cos_newton']:.3f}).
2. **Also positive, but still toy-scoped:** Muon is more Newton-aligned than the simple Adam-like baseline here ({muon['mean_cos_newton']:.3f} vs {adam['mean_cos_newton']:.3f}).
3. **Most cautious comparison:** Muon vs NormSGD is much tighter ({muon['mean_cos_newton']:.3f} vs {norm['mean_cos_newton']:.3f}), so this notebook should present that comparison as **weak / mixed or inconclusive unless the returned seed-level evidence is cleanly one-sided**.
4. **Truthfulness about scope:** the notebook should say these results are **consistent with a direction-quality story in a toy deep-linear model**, not that they generally prove Muon's mechanism.

If a stronger mechanism claim is needed later, the next useful follow-up is a **shared-state control** (evaluate all optimizer directions at the same local state) plus wider or adaptive LR sweeps to reduce boundary-censoring.
'''))
